# Agent 第 06 课：Planning（Plan-and-Execute）

ReAct 是"**想一步做一步**"，简单直接，对大多数短任务够用。但当任务**多步骤、有依赖**时，它有几个毛病：

- 每一步都要重新读一遍完整 context，token 浪费大
- 模型容易"走偏"——中间某步想岔了后面全错
- 没有全局视角，可能做到一半发现前面漏了一步又回头

**Plan-and-Execute** 范式把 agent 拆成两个角色：

```
Planner:   把任务拆成 N 个子步骤（只出计划，不执行）
Executor:  按计划一步一步执行（每步可以用 ReAct 小 agent）
[optional] Replanner: 每执行几步后回看，必要时改计划
```

对应论文：*Plan-and-Solve Prompting*（2023）、*ReWOO*、*LLMCompiler*。这里我们用最简版本跑通概念。

## 1. 什么时候 Plan > ReAct，什么时候 ReAct > Plan

| 场景 | 推荐 |
|---|---|
| 用户提问 1-2 步能答（"今天北京天气？"） | **ReAct** |
| 任务清晰、步骤线性（"下载数据、清洗、出图、写报告"） | **Plan** |
| 高度探索性（每步结果影响下一步该做什么） | **ReAct** |
| 成本敏感（希望少调 LLM） | **Plan**（计划一次，执行照做） |
| 任务需要跟外界频繁交互 | **Hybrid**（Plan 出框架，每步用 ReAct 执行） |

真实系统常常是 **Plan 一次 + 每步内部 ReAct**。我们就实现这个。

In [ ]:
import json, re
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 2. Planner：输出 JSON 计划

和工具调用一样——**要求 Planner 输出结构化 JSON**，这样 Executor 才能自动迭代。

In [ ]:
PLANNER_SYSTEM = """You are a task planner. Given a user goal and a list of available tools, produce a concrete step-by-step plan.

Output strict JSON:
{"plan": ["step 1 ...", "step 2 ...", ...]}

Rules:
- Each step must be a single, executable action described in natural language.
- Steps should be minimal — don't add verification or reporting unless asked.
- Do not execute anything; only plan.
- Keep it under 6 steps."""

def make_plan(goal, tools_desc):
    usr = f'Goal: {goal}\n\nAvailable tools:\n{tools_desc}\n\nReturn JSON only.'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':PLANNER_SYSTEM},
                  {'role':'user','content':usr}])
    txt = r.choices[0].message.content
    m = re.search(r'\{.*\}', txt, re.S)
    return json.loads(m.group(0))['plan']

## 3. Executor：每一步跑一个小 ReAct

把第 03 课的 agent 稍微改一下——给定一个**具体 step**作为任务，它自己用工具完成，然后返回"这一步的输出"。

**关键点**：上一步的输出要作为"上下文"传给下一步，不然每一步是独立的。

In [ ]:
import math, datetime

def t_calc(a): return eval(a['expr'], {'__builtins__':{}}, {k:getattr(math,k) for k in ['sqrt','pi','e','sin','cos','log']})
def t_today(a): return datetime.date.today().isoformat()
def t_days(a):
    d1=datetime.date.fromisoformat(a['date1']); d2=datetime.date.fromisoformat(a['date2'])
    return abs((d2-d1).days)

TOOLS=[
    {'name':'calculator','description':'Math expression','parameters':{'type':'object','properties':{'expr':{'type':'string'}},'required':['expr']}},
    {'name':'today','description':"Today's date",'parameters':{'type':'object','properties':{}}},
    {'name':'days_between','description':'Days between two ISO dates','parameters':{'type':'object','properties':{'date1':{'type':'string'},'date2':{'type':'string'}},'required':['date1','date2']}},
]
IMPL={'calculator':t_calc,'today':t_today,'days_between':t_days}

TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)
FINAL_RE = re.compile(r'Final Answer:\s*(.+)', re.S)

EXEC_SYSTEM = f"""You execute ONE step of a larger plan. You have these tools:
{json.dumps(TOOLS, indent=2)}

Use <tool_call>...</tool_call> to call tools. When the step is done, reply 'Final Answer: <concise result>'."""

def execute_step(step_text, prior_results, max_steps=5, verbose=False):
    context = '\n'.join([f'- {k}: {v}' for k,v in prior_results.items()]) or '(none)'
    usr = f'Step to execute: {step_text}\n\nResults from earlier steps:\n{context}'
    messages = [{'role':'system','content':EXEC_SYSTEM},{'role':'user','content':usr}]
    for _ in range(max_steps):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=messages)
        t = r.choices[0].message.content
        messages.append({'role':'assistant','content':t})
        if verbose: print('  >', t[:200].replace('\n',' '))
        mf = FINAL_RE.search(t)
        if mf: return mf.group(1).strip()
        mt = TOOL_CALL_RE.search(t)
        if mt:
            call = json.loads(mt.group(1)); name=call['name']; args=call.get('arguments') or {}
            try: obs={'result':IMPL[name](args)}
            except Exception as e: obs={'error':str(e)}
            messages.append({'role':'user','content':f'<tool_response>{json.dumps(obs)}</tool_response>'})
        else:
            messages.append({'role':'user','content':'Issue a <tool_call> or Final Answer.'})
    return None

## 4. 主驱动：plan → execute each → final answer

In [ ]:
def plan_and_execute(goal, verbose=True):
    tools_desc = '\n'.join([f"- {t['name']}: {t['description']}" for t in TOOLS])
    plan = make_plan(goal, tools_desc)
    if verbose:
        print('PLAN:')
        for i,s in enumerate(plan,1): print(f'  {i}. {s}')
        print()
    results = {}
    for i, step in enumerate(plan, 1):
        if verbose: print(f'EXECUTING step {i}: {step}')
        out = execute_step(step, results, verbose=verbose)
        results[f'step_{i}'] = out
        if verbose: print(f'  => {out}\n')
    # 汇总
    summary = client.chat.completions.create(
        model=chat_model, temperature=0,
        messages=[{'role':'system','content':'Given a plan and the results of each step, write a concise final answer to the original user goal.'},
                  {'role':'user','content':f'Goal: {goal}\n\nPlan:\n'+'\n'.join(plan)+f'\n\nResults:\n{json.dumps(results, indent=2)}'}]
    ).choices[0].message.content.strip()
    return summary, plan, results

In [ ]:
goal = 'Compute how many full weeks are left from today until 2030-01-01, and what fraction of a year that is.'
answer, plan, results = plan_and_execute(goal)
print('=== FINAL ===')
print(answer)

## 5. 对比跑一下纯 ReAct

同一道题不用 plan，直接 ReAct。看是否差不多（短任务上确实差不多，plan 的优势在长任务才显现）。

In [ ]:
def run_react(goal, max_steps=8):
    sys = f'You are a helpful agent. Tools:\n{json.dumps(TOOLS, indent=2)}\nUse <tool_call>...</tool_call>; end with Final Answer: ...'
    msgs=[{'role':'system','content':sys},{'role':'user','content':goal}]
    for _ in range(max_steps):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=msgs)
        t = r.choices[0].message.content; msgs.append({'role':'assistant','content':t})
        mf=FINAL_RE.search(t); mt=TOOL_CALL_RE.search(t)
        if mf: return mf.group(1).strip()
        if mt:
            c=json.loads(mt.group(1))
            try: obs={'result':IMPL[c['name']](c.get('arguments') or {})}
            except Exception as e: obs={'error':str(e)}
            msgs.append({'role':'user','content':f'<tool_response>{json.dumps(obs)}</tool_response>'})
        else:
            msgs.append({'role':'user','content':'Issue a tool_call or Final Answer.'})
    return None

print('ReAct answer:', run_react(goal))

## 6. Replanner：当计划走偏了怎么办

真实任务里每步可能失败、或结果超出预期，这时候要**重新规划**。模式：

```
执行 step i → 看结果 → 问 replanner："计划还成立吗？要改吗？" → 改完继续
```

我们这一课不实现，留一个思考题：你能加一个 replanner 吗？逻辑：
- 每个 step 执行后判断 `error?` 或 `unexpected_output?`
- 如果是，调用一次 planner 更新剩下的 plan

框架（LangGraph、smolagents）内置了这个，但自己写一遍你会知道它不过是 `while plan_not_done: step; if failed: replan()`。

## 7. 工程直觉

1. **Plan-and-Execute 的核心优势是"可预期性"**，不是聪明。你在执行前就能看到 agent 要做什么，可以审核、可以估算成本、出错可以定位到具体步骤。
2. **Planner 用强模型，Executor 用便宜模型**是经典省钱组合。Plan 决定策略（质量关键），Execute 是重复劳动（便宜就行）。
3. **步骤数量要控制**。计划超过 8-10 步时，模型很容易列出空泛的 "Step 5: verify the results"——这种废步骤必须剔除，不然执行器不知道该做什么。
4. **上一步结果要显式传给下一步**。我们 `prior_results` 字典就是干这个。不传的话每步独立，就变成了并行而不是依赖。
5. **Plan-and-Execute ≠ 好用**。很多任务纯 ReAct 反而更快更便宜。**默认用 ReAct，任务明显多步骤且线性时升级到 Plan**——这是我的经验法则。

---

下一课：**第 07 课 Reflection（自反思）与错误恢复**——让 agent 批评自己、回头重做。最接近人类"检查答案"的那一步。